# Traffic Violation System — Model Training (Google Colab)

Trains the **two** models the pipeline needs (the vehicle/person detector is pretrained COCO YOLO and needs no training):

1. **Helmet detector** (`helmet_yolov8.pt`) — helmet / no-helmet on Indian riders
2. **License-plate detector** (`plate_yolov8.pt`) — Indian number-plate localization

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

At the end you download two `.pt` files → drop them in `models/weights/` on your local machine and the pipeline runs on your GTX 1650.

In [ ]:
# 1. Install dependencies
!pip install -q ultralytics roboflow kaggle
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable T4 in Runtime settings')

## 2. Get the datasets
Pick **one** download path per dataset. Roboflow is easiest (one API key). Fill in your own workspace/project/version if you chose a different dataset — copy them from the Roboflow `Download → YOLOv8` panel.

In [ ]:
# 2a. Roboflow download (helmet + plate)
from roboflow import Roboflow
rf = Roboflow(api_key='PASTE_YOUR_ROBOFLOW_API_KEY')

# --- Helmet ---
helmet_ds = (rf.workspace('vehicles-dataset-zoqwx')
               .project('motorcycle-helmet-detection-dataset-axtx6')
               .version(1).download('yolov8', location='/content/helmet_dataset'))

# --- License plate ---
plate_ds = (rf.workspace('roboflow-universe-projects')
              .project('license-plate-recognition-rxg4e')
              .version(11).download('yolov8', location='/content/plate_dataset'))

print('helmet data.yaml:', helmet_ds.location + '/data.yaml')
print('plate  data.yaml:', plate_ds.location + '/data.yaml')

In [ ]:
# 2b. (ALTERNATIVE) Kaggle download — richer Indian helmet labels
# from google.colab import files; files.upload()   # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d aryanvaid13/indian-helmet-detection-dataset -p /content/helmet_dataset --unzip
# # Then point the train cell at the data.yaml that ships inside the archive.

## 3. Train the helmet detector
`yolo11s`/`yolov8s` is the accuracy-vs-speed sweet spot and runs real-time on a GTX 1650 at inference. T4 trains it comfortably at batch 16.

In [ ]:
from ultralytics import YOLO

helmet_model = YOLO('yolo11s.pt')   # accuracy/speed sweet spot; use 'yolov8s.pt' if you prefer
helmet_model.train(
    data='/content/helmet_dataset/data.yaml',
    epochs=80, imgsz=640, batch=16, device=0,
    patience=20, name='helmet', project='/content/runs',
    augment=True, mosaic=1.0,
)

## 4. Train the license-plate detector

In [ ]:
plate_model = YOLO('yolo11s.pt')
plate_model.train(
    data='/content/plate_dataset/data.yaml',
    epochs=60, imgsz=640, batch=16, device=0,
    patience=15, name='plate', project='/content/runs',
)

## 5. Validate — capture metrics for your report (mAP / P / R)

In [ ]:
for name, m in [('HELMET', helmet_model), ('PLATE', plate_model)]:
    metrics = m.val()
    print(f'{name}: mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}  '
          f'P={metrics.box.mp:.3f}  R={metrics.box.mr:.3f}')

## 6. Download the trained weights
Save these into `models/weights/` locally (filenames must match `configs/pipeline.yaml`).

In [ ]:
import shutil
from google.colab import files
shutil.copy('/content/runs/helmet/weights/best.pt', '/content/helmet_yolov8.pt')
shutil.copy('/content/runs/plate/weights/best.pt',  '/content/plate_yolov8.pt')
files.download('/content/helmet_yolov8.pt')
files.download('/content/plate_yolov8.pt')
print('Done. Place both files in models/weights/ on your machine.')